# 🎬 Studio Video AI - Gerador Neural 3D de Alta Resolução (512px)
### Execução 100% gratuita na GPU NVIDIA T4 do Google Colab

**Instruções:**
1. Verifique se o ambiente está em **GPU T4** (Ambiente de execução -> Alterar tipo de ambiente -> GPU T4).
2. Clique no **Play ▶** abaixo.
3. O processo leva apenas **cerca de 1 minuto**. Ao finalizar, ele vai parar sozinho, **mostrar o vídeo na tela** e **baixar o arquivo .mp4 para o seu computador**!

In [ ]:
# ==============================================================================
# 🚀 RENDERIZADOR NEURAL DIRETO 512PX (SADTALKER 3D)
# ==============================================================================
import os, sys, glob

print('📦 1. Instalando bibliotecas essenciais (sem erros de compilação)...')
# Instalamos apenas pacotes com rodas binárias pré-compiladas (instalação em segundos)
!pip install -q kornia facexlib yacs safetensors edge-tts pydub

%cd /content
if not os.path.exists('/content/SadTalker'):
    !git clone -b v1.1 https://github.com/camenduru/SadTalker

%cd /content/SadTalker

# 2. Correções automáticas de compatibilidade em Python puro
# A) Corrige aviso do NumPy
preprocess_file = '/content/SadTalker/src/face3d/util/preprocess.py'
if os.path.exists(preprocess_file):
    with open(preprocess_file, 'r') as f:
        content = f.read()
    content = content.replace('np.VisibleDeprecationWarning', 'UserWarning')
    with open(preprocess_file, 'w') as f:
        f.write(content)

# B) Corrige importação opcional do face_enhancer para nunca travar
enhancer_file = '/content/SadTalker/src/utils/face_enhancer.py'
if os.path.exists(enhancer_file):
    with open(enhancer_file, 'r') as f:
        enh_content = f.read()
    enh_content = enh_content.replace('from gfpgan import GFPGANer', 'try:\n    from gfpgan import GFPGANer\nexcept Exception:\n    GFPGANer = None')
    with open(enhancer_file, 'w') as f:
        f.write(enh_content)

# 3. Baixa/Verifica pesos neurais 512px do SadTalker
!apt -y install -qq aria2
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/SadTalker_V0.0.2_256.safetensors -d /content/SadTalker/checkpoints -o SadTalker_V0.0.2_256.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/SadTalker_V0.0.2_512.safetensors -d /content/SadTalker/checkpoints -o SadTalker_V0.0.2_512.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/mapping_00109-model.pth.tar -d /content/SadTalker/checkpoints -o mapping_00109-model.pth.tar
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/mapping_00229-model.pth.tar -d /content/SadTalker/checkpoints -o mapping_00229-model.pth.tar

# 4. Baixa a foto do personagem maçônico e o áudio gravado
!curl -s -L -o /content/personagem_maconaria.jpg https://raw.githubusercontent.com/grego921/studio-video-ai/main/personagem_maconaria.jpg
!curl -s -L -o /content/fala_maconica.mp3 https://raw.githubusercontent.com/grego921/studio-video-ai/main/fala_maconica.mp3

# 5. RENDERIZAÇÃO NEURAL DIRETA 512PX (Cerca de 1 minuto na GPU)
print('\n' + '='*70)
print('🎬 RENDERIZANDO VÍDEO 3D COM SINCRONIA LABIAL (512x512)...')
print('='*70 + '\n')

!python inference.py \
  --driven_audio /content/fala_maconica.mp3 \
  --source_image /content/personagem_maconaria.jpg \
  --result_dir /content/resultado \
  --still \
  --preprocess crop \
  --size 512 \
  --batch_size 2

# 6. Exibe o vídeo gerado na tela e baixa para o computador
from google.colab import files
from IPython.display import HTML, display
from base64 import b64encode

videos = glob.glob('/content/resultado/*/*.mp4')
if videos:
    latest_video = max(videos, key=os.path.getctime)
    print('\n' + '='*70)
    print(f'🎉 VÍDEO NEURAL FINALIZADO COM SUCESSO!')
    print(f'📁 Arquivo: {latest_video}')
    print('📥 O download começará automaticamente em instantes...')
    print('='*70 + '\n')
    
    # Exibe no próprio notebook para você assistir na hora
    try:
        video_bytes = open(latest_video, 'rb').read()
        data_url = 'data:video/mp4;base64,' + b64encode(video_bytes).decode()
        display(HTML(f'<video width="480" controls autoplay style="border-radius:12px; margin-top:10px;"><source src="{data_url}" type="video/mp4"></video>'))
    except Exception as e:
        pass
        
    files.download(latest_video)
else:
    print('❌ Erro: O vídeo não foi encontrado na pasta de resultados.')
